In [0]:
dbutils.widgets.text("landing_timestamp","")
dbutils.widgets.text("source_table_name","")
dbutils.widgets.text("merge_keys","")
dbutils.widgets.text("load_type","")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_table_name = dbutils.widgets.get("source_table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("Step 1: Reading notebook parameters")
print(f"  landing_timestamp : {landing_timestamp}")
print(f"  source_table_name : {source_table_name}")
print(f"  merge_keys        : {merge_keys}")
print(f"  load_type         : {load_type}")

In [0]:
source_table = f"caresync_databricks.bronze.{source_table_name}"
print(f"Step 2: Reading bronze table {source_table} for landing_timestamp = '{landing_timestamp}'")

bronze_df = spark.read.table(f"caresync_databricks.bronze.{source_table_name}")\
            .filter(f"landing_timestamp = '{landing_timestamp}'")

print(f"  Bronze data loaded with columns: {bronze_df.columns}")

In [0]:
from pyspark.sql import functions as F
print("Step 3: Adding audit columns insert_timestamp and updated_timestamp")
updated_bronze_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp())\
            .withColumn("updated_timestamp", F.current_timestamp())

print("  Audit columns added successfully")

In [0]:
merge_keys_list = [key.strip() for key in merge_keys.split(",") if key.strip()]
target_table = f"caresync_databricks.silver.{source_table_name}"
print(f"Step 4: Writing data to silver table {target_table} using load_type = {load_type}")

if load_type == "FULL":
    print("  FULL load: overwriting target table")
    updated_bronze_df.write.mode("overwrite").saveAsTable(target_table)
    print("  FULL load completed")

elif load_type == "APPEND":
    print("  APPEND load: appending to target table")
    updated_bronze_df.write.mode("append").saveAsTable(target_table)
    print("  APPEND load completed")

elif load_type == "MERGE":

    print(f"  MERGE load: merge keys = {merge_keys_list}")
    if not spark.catalog.tableExists(target_table):
        print("  Target table does not exist, creating it with initial load")
        updated_bronze_df.write.mode("overwrite").saveAsTable(target_table)
        print("  Initial load completed")

    else:
        print("  Target table exists, preparing MERGE")
        updated_bronze_df.createOrReplaceTempView("silver_merge_source")

        # Build MERGE condition
        merge_conditions = []

        for key in merge_keys_list:
            merge_conditions.append(
                f"target.`{key}` = source.`{key}`"
            )

        merge_condition = " AND ".join(merge_conditions)
        print(f"  Merge condition: {merge_condition}")

        # Build UPDATE assignments
        update_columns = []

        for column in updated_bronze_df.columns:
            if column not in merge_keys_list and column != "insert_timestamp":
                update_columns.append(
                    f"target.`{column}` = source.`{column}`"
                )

        update_assignments = ", ".join(update_columns)
        print(f"  Columns to update on match: {len(update_columns)}")

        # Build INSERT columns and values
        insert_column_list = []
        insert_value_list = []

        for column in updated_bronze_df.columns:
            insert_column_list.append(f"`{column}`")
            insert_value_list.append(f"source.`{column}`")

        insert_columns = ", ".join(insert_column_list)
        insert_values = ", ".join(insert_value_list)
        print(f"  Columns to insert when not matched: {len(insert_column_list)}")

        # Execute MERGE
        print("  Executing MERGE statement")
        spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING silver_merge_source AS source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns}) VALUES ({insert_values})
        """)
        print(" MERGE completed")

else:
    print(f"  WARNING: Unknown load_type '{load_type}'. No data was written.")

In [0]:
print("Step 5: Counting processed records")
record_count = updated_bronze_df.count()
print(f"  Records processed: {record_count}")
print("Bronze to silver load finished, exiting notebook")
dbutils.notebook.exit(str(record_count))